In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
from ipywidgets import widgets
from natsort import natsort_keygen

from latent_dynamics.dayang.activations import Activations, extract_activations
from latent_dynamics.dayang.data import DATASET_REGISTRY, load_dataset_from_spec
from latent_dynamics.dayang.model import MODEL_REGISTRY, load_model_and_tokenizer
from latent_dynamics.dayang.projections import compute_layerwise_pca, plot_layerwise_pca, plot_layerwise_pca_ratio
from latent_dynamics.dayang.scoring import compute_layerwise_score, plot_layerwise_score

torch.set_grad_enabled(False)

print(f"Available models:{'\n - '.join([''] + list(MODEL_REGISTRY))}")
print(f"Available datasets:{'\n - '.join([''] + list(DATASET_REGISTRY))}")

# Manual analysis

In [ ]:
dataset = load_dataset_from_spec("wildjailbreak/train/vanilla", max_samples=200)

In [ ]:
model, tokenizer = load_model_and_tokenizer("gemma3_270m")

In [ ]:
activations = extract_activations(model, tokenizer, dataset)

## Projection analysis

In [ ]:
pcas = compute_layerwise_pca(activations, pool_method="last")
plot_layerwise_pca_ratio(pcas)

In [ ]:
plot_layerwise_pca(activations, pcas, pool_method="last")

## Scoring analysis

In [ ]:
scorers = compute_layerwise_score(activations, method="difference_in_mean")

In [ ]:
plot_layerwise_score(activations, scorers, pool_method="last")

# Interactive analysis

## Projection analysis

In [ ]:
def analyze_layerwise_pca():
    state = {"activations": None, "pcas": {}}

    # Layer 1: Extraction
    models = list(MODEL_REGISTRY.keys())
    datasets = list(DATASET_REGISTRY.keys())

    w_model = widgets.Dropdown(options=models, value=models[0], description="Model")
    w_dataset = widgets.Dropdown(options=datasets, value=datasets[0], description="Dataset")
    w_max_samples = widgets.IntText(value=200, min=1, description="Max samples:")
    w_include_response = widgets.Dropdown(options=[False, True, "Sorry", "Sure"], value=False, description="Include response")
    w_apply_chat_template = widgets.Checkbox(value=False, description="Apply chat template")
    btn_extract = widgets.Button(description="Extract Activations", button_style="primary")
    out_extract = widgets.Output()
    layer1 = widgets.VBox(
        [w_model, w_dataset, w_max_samples, w_include_response, w_apply_chat_template, btn_extract, out_extract]
    )

    # Layer 2: Analysis Setup
    w_pca_safe = widgets.SelectMultiple(description="Safe samples")
    w_pca_unsafe = widgets.SelectMultiple(description="Unsafe samples")
    w_pca_pool = widgets.Dropdown(options=["all", "first", "mid", "last", "mean"], value="last", description="Pool")
    w_pca_bos = widgets.Checkbox(value=False, description="Include BOS")
    col_pca = widgets.VBox([widgets.HTML("<b>PCA Setup</b>"), w_pca_safe, w_pca_unsafe, w_pca_pool, w_pca_bos])
    w_vis_safe = widgets.SelectMultiple(description="Safe samples")
    w_vis_unsafe = widgets.SelectMultiple(description="Unsafe samples")
    w_vis_pool = widgets.Dropdown(options=["all", "first", "mid", "last", "mean"], value="last", description="Pool")
    w_vis_bos = widgets.Checkbox(value=False, description="Include BOS")
    col_vis = widgets.VBox(
        [widgets.HTML("<b>Visualization Setup</b>"), w_vis_safe, w_vis_unsafe, w_vis_pool, w_vis_bos]
    )
    out_plot = widgets.Output()
    layer2 = widgets.VBox([widgets.HBox([col_pca, col_vis]), out_plot])

    # Add interaction handlers for layer 1
    def on_extract(*args):
        btn_extract.disabled = True
        # Run extraction
        with out_extract:
            out_extract.clear_output(wait=True)
            model, tokenizer = load_model_and_tokenizer(w_model.value)
            dataset = load_dataset_from_spec(w_dataset.value, max_samples=w_max_samples.value)
            activations = extract_activations(
                model,
                tokenizer,
                dataset,
                include_response=w_include_response.value,
                apply_chat_template=w_apply_chat_template.value,
            )
        # Update state
        state["activations"] = activations
        state["pcas"] = {}

        # Update layer 2 widgets
        samples_safe = sorted([meta["id"] for meta in activations.metadata.values() if meta["is_safe"]], key=natsort_keygen())
        samples_unsafe = sorted([meta["id"] for meta in activations.metadata.values() if not meta["is_safe"]], key=natsort_keygen())
        w_pca_safe.options = samples_safe
        w_pca_unsafe.options = samples_unsafe
        w_vis_safe.options = samples_safe
        w_vis_unsafe.options = samples_unsafe
        w_pca_safe.value = ()
        w_pca_unsafe.value = ()
        w_vis_safe.value = ()
        w_vis_unsafe.value = ()

        btn_extract.disabled = False

        update_plot()

    btn_extract.on_click(on_extract)

    # Add interaction handlers for layer 2
    def update_plot(*args):
        if state["activations"] is None:
            return

        with out_plot:
            out_plot.clear_output(wait=True)

            # Select samples for PCA
            pca_sample_ids = list(w_pca_safe.value) + list(w_pca_unsafe.value)
            pca_activations = state["activations"].select(pca_sample_ids) if pca_sample_ids else state["activations"]
            # Compute and cache PCA
            cache_key = (tuple(sorted(pca_sample_ids)), w_pca_pool.value, w_pca_bos.value)
            if cache_key not in state["pcas"]:
                state["pcas"][cache_key] = compute_layerwise_pca(
                    pca_activations,
                    pool_method=w_pca_pool.value,
                    include_bos=w_pca_bos.value,
                )
            # Plot PCA explained variance ratio
            plot_layerwise_pca_ratio(state["pcas"][cache_key])

            # Select samples for visualization
            vis_sample_ids = list(w_vis_safe.value) + list(w_vis_unsafe.value)
            vis_activations = state["activations"].select(vis_sample_ids) if vis_sample_ids else state["activations"]
            # Plot activations
            plot_layerwise_pca(
                vis_activations,
                pcas=state["pcas"][cache_key],
                pool_method=w_vis_pool.value,
                include_bos=w_vis_bos.value,
            )

    for w in [w_pca_safe, w_pca_unsafe, w_pca_pool, w_pca_bos, w_vis_safe, w_vis_unsafe, w_vis_pool, w_vis_bos]:
        w.observe(update_plot, names="value")

    # Display widgets
    display(widgets.VBox([layer1, widgets.HTML("<hr>"), layer2]))


analyze_layerwise_pca()